## 0. Setup & Config

Configure dependencies, experiment parameters, bank universe, outputs, logging, style, and reproducibility.

| Cell | What it does | Output |
|---|---|---|
| 0.1 | Install/import/configure experiment | EXPERIMENT, BANKS, folders, logger |

$$r_{i,t}=\log(P_{i,t})-\log(P_{i,t-1})$$

A strict configuration layer prevents magic numbers and makes the PEAD experiment reproducible across Colab, GitHub, and local runs.


In [ ]:
# 0.1 Silent installs, imports, EXPERIMENT, universe, outputs, logging, style
import subprocess
import sys

required_packages = [
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn",
    "requests", "yfinance", "pandas_datareader", "plotly"
]
for package in required_packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except Exception as exc:
        print(f"[WARNING] Failed to install {package}: {exc}")

import os
import json
import logging
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

np.random.seed(42)

EXPERIMENT = {
    "name": "pead_european_banks_ifrs9_full_experiment",
    "start_date": "2016-01-01",
    "end_date": "2026-01-01",
    "target": "drift_60d",
    "horizons": [5, 20, 60],
    "drift_horizons": [5, 20, 60],
    "test_start": "2021-01-01",
    "embargo": 20,
    "n_quantiles": 5,
    "cost_bps": 10.0,
    "models": ["ols", "ridge"],
    "run_ablation": True,
    "run_backtest": True,
    "save_figures": True,
    "feature_blocks": ["controls", "price_factors", "macro_factors", "event_factors"],
    "winsor_low": 0.01,
    "winsor_high": 0.99,
    "min_train_rows": 80,
    "min_test_rows": 10,
    "walk_forward_frequency": "QE",
    "ridge_alpha": 1.0,
    "top_quantile": 0.8,
    "bottom_quantile": 0.2,
    "synthetic_seed": 42,
    "structural_breaks": {"IFRS9_effective": "2018-01-01", "COVID": "2020-03-01"},
}

BANKS = {
    "ISP.MI": {"name": "Intesa Sanpaolo", "country": "Italy"},
    "UCG.MI": {"name": "UniCredit", "country": "Italy"},
    "BNP.PA": {"name": "BNP Paribas", "country": "France"},
    "GLE.PA": {"name": "Societe Generale", "country": "France"},
    "SAN.MC": {"name": "Banco Santander", "country": "Spain"},
    "BBVA.MC": {"name": "BBVA", "country": "Spain"},
    "DBK.DE": {"name": "Deutsche Bank", "country": "Germany"},
    "CBK.DE": {"name": "Commerzbank", "country": "Germany"},
    "BARC.L": {"name": "Barclays", "country": "United Kingdom"},
    "HSBA.L": {"name": "HSBC", "country": "United Kingdom"},
}

OUTPUT_ROOT = Path("output")
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR = OUTPUT_ROOT / "tables"
LOGS_DIR = OUTPUT_ROOT / "logs"
DASHBOARD_DIR = OUTPUT_ROOT / "dashboard"
for directory in [OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR, LOGS_DIR, DASHBOARD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger(EXPERIMENT["name"])
logger.setLevel(logging.INFO)
logger.handlers = []
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler = logging.FileHandler(LOGS_DIR / "experiment.log")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

COLORS = {
    "primary": "#01696f", "accent": "#da7101", "q1": "#c0392b",
    "neutral": "#7a7974", "bg": "#f7f6f2", "blue": "#006494",
    "gold": "#d19900", "purple": "#7a39bb",
}
MODEL_COLORS = {"ols": COLORS["blue"], "ridge": COLORS["primary"], "strategy": COLORS["accent"]}
plt.rcParams["figure.dpi"] = 180
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.facecolor"] = COLORS["bg"]
plt.rcParams["savefig.facecolor"] = COLORS["bg"]
sns.set_theme(style="whitegrid", font_scale=1.05)

PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
logger.info("Experiment initialized | Plotly available=%s", PLOTLY_AVAILABLE)


## 1. Data Ingestion

Load European bank prices and risk factors, preferring the central database/Untitled2 loader and falling back to realistic synthetic data if real data are unavailable.

| Cell | What it does | Output |
|---|---|---|
| 1.1 | Load market panel and factors | prices_raw, ff_factors, data_source_summary |

$$P_{i,t} → \{O,H,L,C,V\}_{i,t}$$

Real market data are preferred. Synthetic fallback is explicitly flagged so downstream inference can distinguish research plumbing from live evidence.


In [ ]:
# 1.1 Load data from Untitled2 DB loader, then APIs, then synthetic fallback
from pead_european_banks_ifrs9.src.pead_data_loader import (
    load_pead_data_from_db,
    normalize_price_panel,
    make_synthetic_bank_prices,
)

def load_file_default(path: Path):
    if path.suffix.lower() in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)

bank_tickers = list(BANKS.keys())
data_source_rows = []
ff_factors = None

try:
    db_root = globals().get("DB_ROOT", None)
    file_map = globals().get("FILE_MAP", None)
    load_file_fn = globals().get("load_file", load_file_default)
    prices_raw, ff_factors = load_pead_data_from_db(
        bank_tickers=bank_tickers,
        start_date=EXPERIMENT["start_date"],
        end_date=EXPERIMENT["end_date"],
        db_root=db_root,
        file_map=file_map,
        load_file=load_file_fn,
        logger=logger,
    )
    data_source_rows.append({"series": "prices", "source": "database_finanziario_untitled2", "synthetic": False, "rows": len(prices_raw)})
except Exception as exc:
    logger.warning("Database loader unavailable; trying web API fallback: %s", exc)
    try:
        if importlib.util.find_spec("yfinance") is None:
            raise RuntimeError("yfinance not installed")
        import yfinance as yf
        frames = []
        for ticker in bank_tickers:
            raw = yf.download(ticker, start=EXPERIMENT["start_date"], end=EXPERIMENT["end_date"], auto_adjust=True, progress=False)
            if raw is None or raw.empty:
                logger.warning("No yfinance rows for %s", ticker)
                continue
            raw = raw.reset_index().rename(columns={"Date": "date", "Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
            raw["ticker"] = ticker
            frames.append(raw[["ticker", "date", "open", "high", "low", "close", "volume"]])
        if not frames:
            raise RuntimeError("No yfinance bank prices downloaded")
        prices_raw = normalize_price_panel(pd.concat(frames, ignore_index=True), logger=logger)
        data_source_rows.append({"series": "prices", "source": "yfinance", "synthetic": False, "rows": len(prices_raw)})
    except Exception as web_exc:
        logger.warning("Real price data failed; generating realistic synthetic panel: %s", web_exc)
        prices_raw = make_synthetic_bank_prices(bank_tickers, EXPERIMENT["start_date"], EXPERIMENT["end_date"], seed=EXPERIMENT["synthetic_seed"])
        data_source_rows.append({"series": "prices", "source": "synthetic_bank_factor_model", "synthetic": True, "rows": len(prices_raw)})

if ff_factors is None:
    try:
        if importlib.util.find_spec("pandas_datareader") is None:
            raise RuntimeError("pandas_datareader not installed")
        import pandas_datareader.data as web
        ff = web.DataReader("F-F_Research_Data_Factors_daily", "famafrench", EXPERIMENT["start_date"], EXPERIMENT["end_date"])[0]
        ff.index = pd.to_datetime(ff.index)
        ff_factors = ff / 100.0
        data_source_rows.append({"series": "risk_factors", "source": "famafrench_web", "synthetic": False, "rows": len(ff_factors)})
    except Exception as exc:
        logger.warning("Risk factors unavailable; creating synthetic market factor: %s", exc)
        dates = pd.bdate_range(EXPERIMENT["start_date"], EXPERIMENT["end_date"])
        rng = np.random.default_rng(EXPERIMENT["synthetic_seed"])
        ff_factors = pd.DataFrame({"Mkt-RF": rng.normal(0.0002, 0.01, len(dates)), "SMB": rng.normal(0, 0.004, len(dates)), "HML": rng.normal(0, 0.004, len(dates)), "RF": 0.00002}, index=dates)
        ff_factors["risk_factors_synthetic"] = True
        data_source_rows.append({"series": "risk_factors", "source": "synthetic_factor_model", "synthetic": True, "rows": len(ff_factors)})

data_source_summary = pd.DataFrame(data_source_rows)
data_source_summary.to_csv(TABLES_DIR / "Table_1_data_source_summary.csv", index=False)
print("DATA SOURCE SUMMARY")
print(data_source_summary.to_string(index=False))
print(f"prices_raw shape={prices_raw.shape} tickers={prices_raw['ticker'].nunique()}")


## 2. Cleaning & Alignment

Clean prices, remove invalid observations, compute lag-safe returns, and align cross-bank market proxy returns.

| Cell | What it does | Output |
|---|---|---|
| 2.1 | Clean prices and align market proxy | prices |

$$r^w_{i,t}=\text{winsorize}(r_{i,t};1\%,99\%)$$

The event-study panel must be built from tradable, positive prices and lag-safe return features to avoid spurious PEAD conclusions.


In [ ]:
# 2.1 Clean prices and align market proxy

def winsorize_series(series, low=EXPERIMENT["winsor_low"], high=EXPERIMENT["winsor_high"]):
    lo, hi = series.quantile([low, high])
    return series.clip(lo, hi)


def clean_prices(raw_prices: pd.DataFrame) -> pd.DataFrame:
    required = {"ticker", "date", "close"}
    missing = required - set(raw_prices.columns)
    if missing:
        raise ValueError(f"prices_raw missing required columns: {sorted(missing)}")
    df = raw_prices.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    invalid = df["close"].le(0) | df["close"].isna()
    if invalid.any():
        logger.warning("Dropping invalid close rows: %s", int(invalid.sum()))
    df = df.loc[~invalid].sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")
    group = df.groupby("ticker", group_keys=False)
    df["ret_1d_raw"] = group["close"].pct_change()
    df["ret_1d"] = df["ret_1d_raw"]
    df["ret_1d_w"] = group["ret_1d"].transform(winsorize_series)
    df["log_close"] = np.log(df["close"])
    df["suspension"] = group["date"].diff().dt.days.gt(7).fillna(False)
    if "prices_synthetic" not in df.columns:
        df["prices_synthetic"] = False
    market_proxy = df.groupby("date")["ret_1d_w"].mean().rename("mkt_ret")
    df = df.join(market_proxy, on="date")
    return df.reset_index(drop=True)

prices = clean_prices(prices_raw)
cleaning_summary = prices.groupby("ticker").agg(N=("date", "size"), start=("date", "min"), end=("date", "max"), suspensions=("suspension", "sum")).reset_index()
cleaning_summary.to_csv(TABLES_DIR / "Table_2_cleaning_summary.csv", index=False)
print(cleaning_summary.to_string(index=False))


## 3. Feature Engineering

Build controls, price factors, and IFRS9-style macro/event features in named blocks matching EXPERIMENT.

| Cell | What it does | Output |
|---|---|---|
| 3.1 | Build feature blocks | feature_panel, feature_metadata |

$$X_{i,t}=f(\mathcal{I}_{t-1})$$

Each feature block uses information known before the event date, matching the zero-look-ahead standard for empirical finance.


In [ ]:
# 3.1 Feature blocks: controls, price_factors, macro_factors, event_factors
FEATURE_BLOCKS = {}

def build_controls_features(df):
    out = df.copy()
    group = out.groupby("ticker", group_keys=False)
    out["lag_ret_1d_raw"] = group["ret_1d_w"].shift(1)
    out["lag_mkt_ret_raw"] = out["mkt_ret"].shift(1)
    out["month_raw"] = out["date"].dt.month
    FEATURE_BLOCKS["controls"] = ["lag_ret_1d", "lag_mkt_ret", "month"]
    return out


def build_price_factors_features(df):
    out = df.copy()
    group = out.groupby("ticker", group_keys=False)
    for window in [5, 20, 60]:
        out[f"mom_{window}d_raw"] = group["close"].pct_change(window).shift(1)
        out[f"vol_{window}d_raw"] = group["ret_1d_w"].transform(lambda x: x.rolling(window).std()).shift(1)
    FEATURE_BLOCKS["price_factors"] = ["mom_5d", "mom_20d", "mom_60d", "vol_5d", "vol_20d", "vol_60d"]
    return out


def build_macro_factors_features(df):
    out = df.copy()
    ff = ff_factors.copy()
    ff.index = pd.to_datetime(ff.index)
    ff_daily = ff.reset_index().rename(columns={"index": "date"})
    out = out.merge(ff_daily, on="date", how="left")
    for col in [c for c in ["Mkt-RF", "SMB", "HML", "RF"] if c in out.columns]:
        out[f"{col}_lag1_raw"] = out[col].shift(1)
    FEATURE_BLOCKS["macro_factors"] = [f"{col}_lag1" for col in ["Mkt-RF", "SMB", "HML", "RF"] if f"{col}_lag1_raw" in out.columns]
    return out


def build_event_factors_features(df):
    out = df.copy()
    rng = np.random.default_rng(EXPERIMENT["synthetic_seed"])
    if "event_flag" not in out.columns:
        event_probability = 0.015
        out["event_flag_synthetic"] = rng.binomial(1, event_probability, len(out))
        out["event_flag_raw"] = out["event_flag_synthetic"]
        logger.warning("No real IFRS9 event table supplied; using synthetic event flags for experiment plumbing")
    else:
        out["event_flag_raw"] = out["event_flag"]
    out["event_intensity_raw"] = out.groupby("ticker")["event_flag_raw"].transform(lambda x: x.rolling(60).sum()).shift(1)
    FEATURE_BLOCKS["event_factors"] = ["event_flag", "event_intensity"]
    return out

feature_panel = prices.copy()
for builder in [build_controls_features, build_price_factors_features, build_macro_factors_features, build_event_factors_features]:
    feature_panel = builder(feature_panel)

raw_feature_cols = [c for c in feature_panel.columns if c.endswith("_raw")]
feature_cols = []
for raw_col in raw_feature_cols:
    base = raw_col[:-4]
    feature_panel[base] = feature_panel[raw_col]
    feature_panel[f"{base}_w"] = winsorize_series(pd.to_numeric(feature_panel[raw_col], errors="coerce"))
    feature_cols.append(f"{base}_w")

feature_metadata = pd.DataFrame([{"block": block, "feature": feature} for block, features in FEATURE_BLOCKS.items() for feature in features])
feature_metadata.to_csv(TABLES_DIR / "Table_3_feature_metadata.csv", index=False)
print(feature_metadata.to_string(index=False))


## 4. Targets & Labels

Construct forward drift targets and classification labels over multiple horizons.

| Cell | What it does | Output |
|---|---|---|
| 4.1 | Build PEAD targets | model_panel |

$$DRIFT_{i,t,h}=\sum_{k=1}^{h} r^w_{i,t+k}$$

PEAD is measured as post-event drift after the event date; labels turn continuous drift into ranking/classification tasks.


In [ ]:
# 4.1 Forward drift targets and labels
model_panel = feature_panel.sort_values(["ticker", "date"]).copy()
for horizon in EXPERIMENT["drift_horizons"]:
    forward_sum = model_panel.groupby("ticker")["ret_1d_w"].transform(lambda x: x.shift(-1).rolling(horizon).sum().shift(-(horizon - 1)))
    model_panel[f"drift_{horizon}d"] = forward_sum

target_col = EXPERIMENT["target"]
model_panel["y_binary"] = model_panel.groupby("date")[target_col].transform(lambda x: (x > x.median()).astype(float) if x.notna().sum() >= 2 else np.nan)
model_panel["y_top_q"] = model_panel.groupby("date")[target_col].transform(lambda x: (x >= x.quantile(EXPERIMENT["top_quantile"])).astype(float) if x.notna().sum() >= EXPERIMENT["n_quantiles"] else np.nan)
target_summary = model_panel[["ticker", "date", target_col, "y_binary", "y_top_q"]].dropna(subset=[target_col]).groupby("ticker").agg(N=(target_col, "size"), mean_drift=(target_col, "mean"), std_drift=(target_col, "std")).reset_index()
target_summary.to_csv(TABLES_DIR / "Table_4_target_summary.csv", index=False)
print(target_summary.round(4).to_string(index=False))


## 5. Descriptive Stats

Summarize sample size, coverage, drift distributions, and feature missingness.

| Cell | What it does | Output |
|---|---|---|
| 5.1 | Create descriptive tables | Table_I_summary.csv, Table_feature_missingness.csv |

$$\bar{r}=N^{-1}\sum_i r_i$$

Descriptive diagnostics show whether the sample is sufficiently balanced across banks, periods, and event types before modeling.


In [ ]:
# 5.1 Descriptive stats and missingness by feature block
summary = pd.DataFrame([
    {"metric": "rows", "value": len(model_panel), "N": len(model_panel)},
    {"metric": "tickers", "value": model_panel["ticker"].nunique(), "N": len(model_panel)},
    {"metric": "start", "value": str(model_panel["date"].min().date()), "N": len(model_panel)},
    {"metric": "end", "value": str(model_panel["date"].max().date()), "N": len(model_panel)},
    {"metric": "target_non_null", "value": int(model_panel[target_col].notna().sum()), "N": len(model_panel)},
])
summary.to_csv(TABLES_DIR / "Table_I_summary.csv", index=False)

missing_rows = []
for block, features in FEATURE_BLOCKS.items():
    for feature in features:
        col = f"{feature}_w" if f"{feature}_w" in model_panel.columns else feature
        if col in model_panel.columns:
            missing_rows.append({"block": block, "feature": col, "pct_missing": model_panel[col].isna().mean(), "N": len(model_panel)})
feature_missingness = pd.DataFrame(missing_rows)
feature_missingness.to_csv(TABLES_DIR / "Table_feature_missingness.csv", index=False, float_format="%.4f")
print(summary.to_string(index=False))
print(feature_missingness.round(4).to_string(index=False))


## 6. Exploratory / Event Study

Estimate event-time average abnormal returns and cumulative drift profiles.

| Cell | What it does | Output |
|---|---|---|
| 6.1 | Build event-study curves | Figure_1_event_study.png, Table_II_event_study.csv |

$$CAR(\tau)=\sum_{s=0}^{\tau} AR_s$$

A visual event study provides the first empirical check for delayed market reaction after IFRS9-style credit events.


In [ ]:
# 6.1 Event-study curves around synthetic/real event flags
window = 20
event_rows = model_panel.loc[model_panel["event_flag"].fillna(0).eq(1), ["ticker", "date"]].copy()
event_curve_rows = []
for _, event in event_rows.iterrows():
    series = model_panel[model_panel["ticker"].eq(event["ticker"])].sort_values("date")
    loc = series.index[series["date"].eq(event["date"])]
    if len(loc) == 0:
        continue
    pos = series.index.get_loc(loc[0])
    sample = series.iloc[max(0, pos - window): min(len(series), pos + window + 1)].copy()
    sample["event_time"] = np.arange(len(sample)) - min(window, pos)
    sample["event_date"] = event["date"]
    event_curve_rows.append(sample[["event_time", "ret_1d_w", "event_date", "ticker"]])
if event_curve_rows:
    event_study_panel = pd.concat(event_curve_rows, ignore_index=True)
    event_curve = event_study_panel.groupby("event_time")["ret_1d_w"].agg(["mean", "count"]).reset_index().rename(columns={"mean": "avg_abnormal_return", "count": "N"})
    event_curve["car"] = event_curve["avg_abnormal_return"].cumsum()
else:
    logger.warning("No event rows available; event curve will be empty")
    event_curve = pd.DataFrame(columns=["event_time", "avg_abnormal_return", "N", "car"])
event_curve.to_csv(TABLES_DIR / "Table_II_event_study.csv", index=False, float_format="%.4f")

plt.figure(figsize=(10, 5))
if not event_curve.empty:
    plt.plot(event_curve["event_time"], event_curve["car"], color=COLORS["primary"], label="CAR")
plt.axhline(0, color="black", linestyle="--", lw=0.8)
plt.axvline(0, color=COLORS["accent"], linestyle="--", lw=0.8, label="event")
plt.title("Event-study cumulative abnormal return", fontweight="bold")
plt.xlabel("Event time (trading days)")
plt.ylabel("CAR (return units)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_1_event_study.png")
plt.show()
print(event_curve.head(10).round(4).to_string(index=False))


## 7. Single-Factor Diagnostics

Evaluate each feature block and feature one at a time against the PEAD target.

| Cell | What it does | Output |
|---|---|---|
| 7.1 | Single-factor IC diagnostics | Table_III_single_factor.csv |

$$IC_j=\rho(X_j,y)$$

Rank correlations identify simple monotonic relationships before moving to multivariate econometrics or ML.


In [ ]:
# 7.1 Single-factor diagnostics
single_factor_rows = []
for block, features in FEATURE_BLOCKS.items():
    for feature in features:
        col = f"{feature}_w" if f"{feature}_w" in model_panel.columns else feature
        if col not in model_panel.columns:
            continue
        valid = model_panel[[col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(valid) < EXPERIMENT["min_test_rows"]:
            continue
        single_factor_rows.append({"block": block, "feature": col, "ic_spearman": valid[col].corr(valid[target_col], method="spearman"), "ic_pearson": valid[col].corr(valid[target_col]), "N": len(valid)})
single_factor = pd.DataFrame(single_factor_rows).sort_values("ic_spearman", key=lambda s: s.abs(), ascending=False)
single_factor.to_csv(TABLES_DIR / "Table_III_single_factor.csv", index=False, float_format="%.4f")
print(single_factor.round(4).to_string(index=False))


## 8. Statistical Models (regressions / econometrics)

Fit chronological OLS-style regressions with robust reporting on train/test split.

| Cell | What it does | Output |
|---|---|---|
| 8.1 | Run econometric baseline | Table_IV_regression.csv |

$$y_{i,t}=\alpha+X_{i,t}\beta+\epsilon_{i,t}$$

Econometric baselines remain interpretable and establish whether features explain drift beyond controls.


In [ ]:
# 8.1 Chronological regression baseline
all_features = [f"{feature}_w" for features in FEATURE_BLOCKS.values() for feature in features if f"{feature}_w" in model_panel.columns]
regression_data = model_panel.dropna(subset=all_features + [target_col]).sort_values("date").copy()
test_start = pd.Timestamp(EXPERIMENT["test_start"])
train = regression_data[regression_data["date"] < test_start]
test = regression_data[regression_data["date"] >= test_start]
reg_rows = []
if len(train) >= EXPERIMENT["min_train_rows"] and len(test) >= EXPERIMENT["min_test_rows"]:
    scaler = StandardScaler().fit(train[all_features])
    model = LinearRegression().fit(scaler.transform(train[all_features]), train[target_col])
    pred = model.predict(scaler.transform(test[all_features]))
    reg_rows.append({"model": "ols_chronological", "rmse": np.sqrt(mean_squared_error(test[target_col], pred)), "mae": mean_absolute_error(test[target_col], pred), "r2": r2_score(test[target_col], pred), "N": len(test)})
else:
    logger.warning("Insufficient rows for OLS baseline: train=%s test=%s", len(train), len(test))
    reg_rows.append({"model": "ols_chronological", "rmse": np.nan, "mae": np.nan, "r2": np.nan, "N": len(test)})
regression_results = pd.DataFrame(reg_rows)
regression_results.to_csv(TABLES_DIR / "Table_IV_regression.csv", index=False, float_format="%.4f")
print(regression_results.round(4).to_string(index=False))


## 9. ML Walk-Forward

Run chronological walk-forward ML with embargo and StandardScaler fit only on train folds.

| Cell | What it does | Output |
|---|---|---|
| 9.1 | Walk-forward Ridge model | Table_V_walk_forward.csv, predictions |

$$\hat{f}_t=\arg\min_f \sum_{s<t}(y_s-f(X_s))^2+\lambda||\beta||_2^2$$

Walk-forward validation mimics production deployment and prevents random-split leakage.


In [ ]:
# 9.1 Walk-forward Ridge with embargo
wf_data = regression_data.copy()
fold_rows = []
prediction_rows = []
test_dates = pd.date_range(EXPERIMENT["test_start"], EXPERIMENT["end_date"], freq=EXPERIMENT["walk_forward_frequency"])
for fold_id, period_end in enumerate(test_dates, start=1):
    period_start = period_end - pd.offsets.QuarterEnd(startingMonth=period_end.month)
    test_mask = (wf_data["date"] > period_start) & (wf_data["date"] <= period_end)
    test_fold = wf_data[test_mask]
    embargo_start = period_start - pd.Timedelta(days=EXPERIMENT["embargo"])
    train_fold = wf_data[wf_data["date"] < embargo_start]
    if len(train_fold) < EXPERIMENT["min_train_rows"] or len(test_fold) < EXPERIMENT["min_test_rows"]:
        logger.warning("Skipping fold %s | train=%s test=%s", fold_id, len(train_fold), len(test_fold))
        continue
    scaler = StandardScaler().fit(train_fold[all_features])
    model = Ridge(alpha=EXPERIMENT["ridge_alpha"]).fit(scaler.transform(train_fold[all_features]), train_fold[target_col])
    pred = model.predict(scaler.transform(test_fold[all_features]))
    fold_rmse = np.sqrt(mean_squared_error(test_fold[target_col], pred))
    fold_rows.append({"fold": fold_id, "test_start": test_fold["date"].min(), "test_end": test_fold["date"].max(), "rmse": fold_rmse, "mae": mean_absolute_error(test_fold[target_col], pred), "N": len(test_fold)})
    tmp = test_fold[["date", "ticker", target_col]].copy()
    tmp["prediction"] = pred
    tmp["fold"] = fold_id
    prediction_rows.append(tmp)
walk_forward_metrics = pd.DataFrame(fold_rows)
walk_forward_predictions = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame(columns=["date", "ticker", target_col, "prediction", "fold"])
walk_forward_metrics.to_csv(TABLES_DIR / "Table_V_walk_forward.csv", index=False, float_format="%.4f")
walk_forward_predictions.to_csv(TABLES_DIR / "Table_V_walk_forward_predictions.csv", index=False, float_format="%.4f")
print(walk_forward_metrics.round(4).to_string(index=False))


## 10. Feature Ablation

Run block-level ablation against controls-only and plot delta performance.

| Cell | What it does | Output |
|---|---|---|
| 10.1 | Ablation by feature block | Table_ablation.csv, Figure_3_ablation.png |

$$\Delta_b=Metric(Controls+b)-Metric(Controls)$$

Ablation reveals which economic feature families add incremental value.


In [ ]:
# 10.1 Feature block ablation

def evaluate_feature_set(features):
    data = model_panel.dropna(subset=features + [target_col]).sort_values("date")
    train = data[data["date"] < pd.Timestamp(EXPERIMENT["test_start"])]
    test = data[data["date"] >= pd.Timestamp(EXPERIMENT["test_start"])]
    if len(train) < EXPERIMENT["min_train_rows"] or len(test) < EXPERIMENT["min_test_rows"]:
        return np.nan, len(test)
    scaler = StandardScaler().fit(train[features])
    model = Ridge(alpha=EXPERIMENT["ridge_alpha"]).fit(scaler.transform(train[features]), train[target_col])
    pred = model.predict(scaler.transform(test[features]))
    return -np.sqrt(mean_squared_error(test[target_col], pred)), len(test)

controls = [f"{f}_w" for f in FEATURE_BLOCKS.get("controls", []) if f"{f}_w" in model_panel.columns]
baseline_metric, baseline_n = evaluate_feature_set(controls)
ablation_rows = [{"block": "controls", "metric_neg_rmse": baseline_metric, "delta_vs_controls": 0.0, "N": baseline_n}]
for block, features in FEATURE_BLOCKS.items():
    if block == "controls":
        continue
    active = controls + [f"{f}_w" for f in features if f"{f}_w" in model_panel.columns]
    metric, n = evaluate_feature_set(active)
    ablation_rows.append({"block": block, "metric_neg_rmse": metric, "delta_vs_controls": metric - baseline_metric if pd.notna(metric) and pd.notna(baseline_metric) else np.nan, "N": n})
ablation = pd.DataFrame(ablation_rows).sort_values("delta_vs_controls", ascending=True)
ablation.to_csv(TABLES_DIR / "Table_ablation.csv", index=False, float_format="%.4f")

plt.figure(figsize=(10, 5))
plot_df = ablation.dropna(subset=["delta_vs_controls"])
plt.barh(plot_df["block"], plot_df["delta_vs_controls"], color=COLORS["primary"])
plt.axvline(0, color="black", linestyle="--", lw=0.8)
plt.title("Feature block ablation vs controls", fontweight="bold")
plt.xlabel("Delta metric (negative RMSE)")
plt.ylabel("Feature block")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_3_ablation.png")
plt.show()
print(ablation.round(4).to_string(index=False))


## 11. Backtest / Strategy Evaluation

Translate predictions into a simple long/short PEAD strategy with transaction costs.

| Cell | What it does | Output |
|---|---|---|
| 11.1 | Strategy evaluation | Table_VI_backtest.csv, Figure_4_equity_curve.png |

$$r^{LS}_t=\bar{r}_{top,t}-\bar{r}_{bottom,t}-c_t$$

A trading diagnostic connects statistical predictability to portfolio relevance.


In [ ]:
# 11.1 Portfolio-style PEAD strategy evaluation
bt_rows = []
if not walk_forward_predictions.empty:
    for date, group in walk_forward_predictions.groupby("date"):
        group = group.dropna(subset=["prediction", target_col])
        if len(group) < EXPERIMENT["n_quantiles"]:
            continue
        high = group["prediction"].quantile(EXPERIMENT["top_quantile"])
        low = group["prediction"].quantile(EXPERIMENT["bottom_quantile"])
        long_ret = group.loc[group["prediction"] >= high, target_col].mean()
        short_ret = group.loc[group["prediction"] <= low, target_col].mean()
        turnover_cost = EXPERIMENT["cost_bps"] / 10000.0
        bt_rows.append({"date": date, "long_return": long_ret, "short_return": short_ret, "long_short_return": long_ret - short_ret - turnover_cost, "N": len(group)})
backtest = pd.DataFrame(bt_rows).sort_values("date")
if not backtest.empty:
    backtest["equity_curve"] = (1 + backtest["long_short_return"].fillna(0)).cumprod()
backtest.to_csv(TABLES_DIR / "Table_VI_backtest.csv", index=False, float_format="%.4f")

plt.figure(figsize=(10, 5))
if not backtest.empty:
    plt.plot(backtest["date"], backtest["equity_curve"], color=COLORS["primary"], label="PEAD long/short")
plt.axhline(1, color="black", linestyle="--", lw=0.8)
plt.title("PEAD strategy equity curve", fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Growth of 1 (net of costs)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_4_equity_curve.png")
plt.show()
print(backtest.tail(10).round(4).to_string(index=False))


## 12. Interpretability

Explain model outputs through coefficients and feature importance summaries.

| Cell | What it does | Output |
|---|---|---|
| 12.1 | Interpretability tables | Table_VII_interpretability.csv, Figure_5_importance.png |

$$Importance_j=|\beta_j|$$

Transparent drivers help distinguish economically meaningful PEAD signals from accidental correlations.


In [ ]:
# 12.1 Interpretability from Ridge coefficients
importance_rows = []
if len(train) >= EXPERIMENT["min_train_rows"] and all_features:
    scaler = StandardScaler().fit(train[all_features])
    model = Ridge(alpha=EXPERIMENT["ridge_alpha"]).fit(scaler.transform(train[all_features]), train[target_col])
    for feature, coef in zip(all_features, model.coef_):
        block = next((block for block, features in FEATURE_BLOCKS.items() if feature.replace("_w", "") in features), "unknown")
        importance_rows.append({"feature": feature, "block": block, "coefficient": coef, "importance_abs": abs(coef), "N": len(train)})
importance = pd.DataFrame(importance_rows).sort_values("importance_abs", ascending=False) if importance_rows else pd.DataFrame(columns=["feature", "block", "coefficient", "importance_abs", "N"])
importance.to_csv(TABLES_DIR / "Table_VII_interpretability.csv", index=False, float_format="%.4f")

plt.figure(figsize=(10, 5))
plot_imp = importance.head(15).iloc[::-1]
plt.barh(plot_imp["feature"], plot_imp["importance_abs"], color=COLORS["purple"])
plt.title("Top model drivers", fontweight="bold")
plt.xlabel("Absolute standardized coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_5_importance.png")
plt.show()
print(importance.head(15).round(4).to_string(index=False))


## 13. Robustness Checks

Run subperiod, placebo, and sensitivity checks.

| Cell | What it does | Output |
|---|---|---|
| 13.1 | Robustness battery | Table_VII_robustness.csv, Figure_6_sensitivity_heatmap.png |

$$H_0:Metric_{true}\le Metric_{placebo}$$

Robustness checks test whether results survive time splits, randomized targets, and transaction-cost assumptions.


In [ ]:
# 13.1 Subperiod, placebo, and sensitivity robustness
robust_rows = []
mid_date = model_panel["date"].quantile(0.5)
for label, subset in [("early", model_panel[model_panel["date"] <= mid_date]), ("late", model_panel[model_panel["date"] > mid_date])]:
    valid = subset.dropna(subset=[target_col])
    robust_rows.append({"test": "subperiod_mean_drift", "variant": label, "metric": valid[target_col].mean(), "N": len(valid)})

placebo = regression_data.copy()
rng = np.random.default_rng(EXPERIMENT["synthetic_seed"])
placebo[target_col] = rng.permutation(placebo[target_col].to_numpy())
if all_features and len(placebo) >= EXPERIMENT["min_train_rows"]:
    train_p = placebo[placebo["date"] < pd.Timestamp(EXPERIMENT["test_start"])]
    test_p = placebo[placebo["date"] >= pd.Timestamp(EXPERIMENT["test_start"])]
    if len(train_p) >= EXPERIMENT["min_train_rows"] and len(test_p) >= EXPERIMENT["min_test_rows"]:
        scaler_p = StandardScaler().fit(train_p[all_features])
        model_p = Ridge(alpha=EXPERIMENT["ridge_alpha"]).fit(scaler_p.transform(train_p[all_features]), train_p[target_col])
        pred_p = model_p.predict(scaler_p.transform(test_p[all_features]))
        robust_rows.append({"test": "placebo_target_shuffle", "variant": "ridge", "metric": -np.sqrt(mean_squared_error(test_p[target_col], pred_p)), "N": len(test_p)})

sensitivity_rows = []
for horizon in EXPERIMENT["drift_horizons"]:
    for cost_bps in [0.0, EXPERIMENT["cost_bps"], EXPERIMENT["cost_bps"] * 2]:
        col = f"drift_{horizon}d"
        valid = model_panel.dropna(subset=[col])
        sensitivity_rows.append({"horizon": horizon, "cost_bps": cost_bps, "metric": valid[col].mean() - cost_bps / 10000.0, "N": len(valid)})
robustness = pd.DataFrame(robust_rows)
sensitivity = pd.DataFrame(sensitivity_rows)
robustness_full = pd.concat([robustness, sensitivity.rename(columns={"horizon": "variant"}).assign(test="sensitivity_horizon_cost")], ignore_index=True, sort=False)
robustness_full.to_csv(TABLES_DIR / "Table_VII_robustness.csv", index=False, float_format="%.4f")

heatmap_data = sensitivity.pivot(index="horizon", columns="cost_bps", values="metric")
plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="RdYlGn", center=0)
plt.title("Sensitivity: horizon × cost", fontweight="bold")
plt.xlabel("Cost (bps)")
plt.ylabel("Horizon (days)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_6_sensitivity_heatmap.png")
plt.show()
print(robustness_full.round(4).to_string(index=False))


## 14. Final Summary

Create a shareable dashboard and final artifact inventory.

| Cell | What it does | Output |
|---|---|---|
| 14.1 | Build dashboard and print final artifact counts | HTML dashboard, artifact list |

$$Report=\{Tables,Figures,Diagnostics,Summary\}$$

The final section packages the experiment into review-ready artifacts for GitHub, Drive, advisors, and reviewers.


In [ ]:
# 14.1 Build lightweight HTML dashboard
summary_html = summary.to_html(index=False)
ablation_html = ablation.to_html(index=False, float_format=lambda x: f"{x:.4f}")
wf_html = walk_forward_metrics.to_html(index=False, float_format=lambda x: f"{x:.4f}")
robust_html = robustness_full.to_html(index=False, float_format=lambda x: f"{x:.4f}")
html = f"""
<html><head><title>PEAD European Banks IFRS9 Dashboard</title>
<style>
body {{font-family: Arial, sans-serif; background:{COLORS['bg']}; color:#1f2933; margin:24px;}}
.header {{background:{COLORS['primary']}; color:white; padding:20px; border-radius:14px;}}
.card {{display:inline-block; background:white; margin:10px; padding:14px; border-left:5px solid {COLORS['accent']}; border-radius:10px; min-width:180px;}}
.section {{background:white; margin:16px 0; padding:18px; border-radius:12px;}}
table {{border-collapse: collapse; width:100%; font-size:13px;}}
th {{background:{COLORS['primary']}; color:white; padding:6px;}}
td {{border-bottom:1px solid #ddd; padding:6px;}}
</style></head><body>
<div class='header'><h1>PEAD European Banks IFRS9 Full Experiment</h1><p>Market data ingestion, event-study, walk-forward ML, ablation, strategy evaluation, and robustness.</p></div>
<div class='card'><b>Rows</b><br>{len(model_panel):,}</div>
<div class='card'><b>Banks</b><br>{model_panel['ticker'].nunique():,}</div>
<div class='card'><b>Target</b><br>{target_col}</div>
<div class='card'><b>WF folds</b><br>{len(walk_forward_metrics):,}</div>
<div class='section'><h2>Summary</h2>{summary_html}</div>
<div class='section'><h2>Walk-forward metrics</h2>{wf_html}</div>
<div class='section'><h2>Ablation</h2>{ablation_html}</div>
<div class='section'><h2>Robustness</h2>{robust_html}</div>
</body></html>
"""
dashboard_path = DASHBOARD_DIR / "pead_european_banks_ifrs9_dashboard.html"
dashboard_path.write_text(html, encoding="utf-8")
manifest = {"dashboard": str(dashboard_path), "tables": len(list(TABLES_DIR.glob("*.csv"))), "figures": len(list(FIGURES_DIR.glob("*.png")))}
Path(TABLES_DIR / "final_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Dashboard saved: {dashboard_path}")


In [ ]:
import os
from pathlib import Path
tables  = sorted(Path("output/tables").glob("*.csv"))
figures = sorted(Path("output/figures").glob("*.png"))
print(f"✅ EXPERIMENT COMPLETE")
print(f"   Tables : {len(tables)}")
print(f"   Figures: {len(figures)}")
for f in tables:  print(f"   📋 {f.name}")
for f in figures: print(f"   📊 {f.name}")
